# Markov Regime-Switching Model — Test Simulation

3-process model (each with 3 regimes, transition prob = 1/120 per off-diagonal):
1. **Factor** — scalar (d=1), stochastic, mu = [7, 0, -7]
2. **Volume** — scalar (d=1), deterministic, mu = [1.2, 1, 0.8]
3. **Alpha** — scalar (d=1), deterministic, mu = [0.2, 0, -0.2]

1000 paths, 600 monthly timesteps (50 years), dt=1 (month)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from markov_regimes import RegimeConfig, ProcessConfig, ScenarioModel

%matplotlib inline

## Process Configurations

In [ ]:
# Transition matrix: 1/120 probability of switching to each other regime per month
p_switch = 1 / 120
tm = np.array([
    [1 - 2*p_switch, p_switch, p_switch],
    [p_switch, 1 - 2*p_switch, p_switch],
    [p_switch, p_switch, 1 - 2*p_switch],
])

# Process 1: Factor process (d=1, 3 regimes, stochastic)
factor_process = ProcessConfig(
    name="Factor",
    d=1,
    transition_matrix=tm,
    regimes=[
        RegimeConfig(K=np.array([[0.5]]), mu=np.array([7.0]),  sigma=np.array([[1.0]])),
        RegimeConfig(K=np.array([[0.5]]), mu=np.array([0.0]),  sigma=np.array([[1.0]])),
        RegimeConfig(K=np.array([[0.5]]), mu=np.array([-7.0]), sigma=np.array([[1.0]])),
    ],
)

In [ ]:
# Process 2: Volume process (d=1, 3 regimes, deterministic — sigma=0)
volume_process = ProcessConfig(
    name="Volume",
    d=1,
    transition_matrix=tm,
    regimes=[
        RegimeConfig(K=np.array([[0.5]]), mu=np.array([1.2]), sigma=np.array([[0.0]])),
        RegimeConfig(K=np.array([[0.5]]), mu=np.array([1.0]), sigma=np.array([[0.0]])),
        RegimeConfig(K=np.array([[0.5]]), mu=np.array([0.8]), sigma=np.array([[0.0]])),
    ],
)

In [ ]:
# Process 3: Alpha process (d=1, 3 regimes, deterministic — sigma=0)
alpha_process = ProcessConfig(
    name="Alpha",
    d=1,
    transition_matrix=tm,
    regimes=[
        RegimeConfig(K=np.array([[0.5]]), mu=np.array([0.2]),  sigma=np.array([[0.0]])),
        RegimeConfig(K=np.array([[0.5]]), mu=np.array([0.0]),  sigma=np.array([[0.0]])),
        RegimeConfig(K=np.array([[0.5]]), mu=np.array([-0.2]), sigma=np.array([[0.0]])),
    ],
)

## Stationary Distributions

Initial regimes are sampled from each process's stationary distribution (left eigenvector of the transition matrix for eigenvalue 1).

In [ ]:
for proc in [factor_process, volume_process, alpha_process]:
    pi = proc.stationary_distribution
    print(f"{proc.name:8s}  pi = [{', '.join(f'{p:.4f}' for p in pi)}]")

## Build Model & Simulate

In [ ]:
model = ScenarioModel(processes=[factor_process, volume_process, alpha_process])

M, T, dt = 1000, 600, 1.0  # 1000 paths, 600 months, dt=1 month
values, regimes = model.simulate(n_paths=M, n_steps=T, dt=dt, seed=42)

print(f"values  shape: {values.shape}  (M={M}, T={T}, D={model.total_dim})")
print(f"regimes shape: {regimes.shape}  (M={M}, T={T}, N={len(model.processes)})")

# Show initial regime distribution vs stationary distribution
print("\nInitial regime frequencies vs stationary distribution:")
for j, proc in enumerate(model.processes):
    init = regimes[:, 0, j]
    freqs = np.bincount(init, minlength=proc.n_regimes) / M
    pi = proc.stationary_distribution
    print(f"  {proc.name:8s}  observed={np.array2string(freqs, precision=3)}  "
          f"stationary={np.array2string(pi, precision=3)}")

## Plot Sample Paths (10 paths per process/factor)

In [ ]:
n_sample = 10
time_axis = np.arange(T)  # months

# Build plot specs: (title, value_column_index, regime_process_index, factor_within_process)
plot_specs = []
col = 0
for j, proc in enumerate(model.processes):
    for f in range(proc.d):
        label = f"{proc.name} factor {f}" if proc.d > 1 else proc.name
        plot_specs.append((label, col + f, j, f))
    col += proc.d

n_plots = len(plot_specs)
fig, axes = plt.subplots(n_plots, 2, figsize=(14, 3.5 * n_plots), squeeze=False)

for row, (title, val_col, reg_idx, factor_idx) in enumerate(plot_specs):
    ax_val = axes[row, 0]
    ax_reg = axes[row, 1]

    proc = model.processes[reg_idx]
    # Map regime index -> mu value for this factor
    regime_mus = np.array([proc.regimes[r].mu[factor_idx] for r in range(proc.n_regimes)])

    for i in range(n_sample):
        ax_val.plot(time_axis, values[i, :, val_col], linewidth=0.8, alpha=0.7)
        # Plot the regime mean being reverted to instead of the regime index
        mu_path = regime_mus[regimes[i, :, reg_idx]]
        ax_reg.step(time_axis, mu_path, linewidth=0.8, alpha=0.7, where="post")

    ax_val.set_title(f"{title} — process value")
    ax_val.set_xlabel("month")
    ax_val.set_ylabel("value")

    ax_reg.set_title(f"{title} — regime mean (μ)")
    ax_reg.set_xlabel("month")
    ax_reg.set_ylabel("μ")
    ax_reg.set_yticks(sorted(regime_mus))

fig.tight_layout()
plt.show()